# 05 Soil Survey (SSURGO)

**Series:** Tribal Soils and Geology

## USDA SSURGO on Pine Ridge and Rosebud

The Soil Survey Geographic Database (SSURGO) is the most detailed publicly
available soil mapping for the United States. For Pine Ridge and Rosebud,
SSURGO provides map units at roughly 1:24,000 scale, with soil component
and horizon data attached to each polygon.

However, SSURGO data quality varies significantly across the reservations.
The soil surveys covering Pine Ridge (Oglala Lakota, Bennett, and Jackson
Counties) and Rosebud (Todd, Mellette, Tripp, and Gregory Counties) were
conducted at different times with different sampling densities. On reservation
lands in particular, sampling intensity was sometimes lower than on adjacent
non-Tribal agricultural lands.

**Required data:** Download SSURGO by county from the ESRI Soil Data
Downloader (https://websoilsurvey.nrcs.usda.gov/) and place in
`data/raw/ssurgo/`. Counties needed:
- Pine Ridge: SD113 (Oglala Lakota), SD007 (Bennett), SD063 (Jackson)
- Rosebud: SD121 (Todd), SD095 (Mellette), SD123 (Tripp), SD055 (Gregory)

In [ ]:
# Imports
import sys
from pathlib import Path
REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path: sys.path.insert(0, str(REPO_ROOT))
import warnings, numpy as np, pandas as pd
import geopandas as gpd, matplotlib.pyplot as plt
import matplotlib.patches as mpatches, contextily as ctx, yaml
from shapely.geometry import box
from src.constants import (
    CRS_GEOGRAPHIC, CRS_PROJECTED, CRS_WEB, REPO_ROOT as _REPO_ROOT,
    OUTPUTS_DIR, FIGURES_DIR, PINE_RIDGE_BBOX, ROSEBUD_BBOX,
    COMBINED_BBOX, STUDY_BBOX, WSD_3D_MODEL, WSD_STRATIGRAPHY, WSD_KEY_UNITS,
)
from src.loaders import load_tribal_boundaries
from src.sovereignty import print_data_acknowledgment, generate_citations, attach_provenance
warnings.filterwarnings("ignore", category=FutureWarning)
%matplotlib inline
with open(_REPO_ROOT/"config"/"config.yaml") as f: CONFIG = yaml.safe_load(f)
TEAL="#007A6E"; TEAL_LT="#E0F4F2"; GRAY="#566573"; TERRACOTTA="#C0392B"
def despine(ax):
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
primary = load_tribal_boundaries(["Pine Ridge","Rosebud"])
print(f"Ready. Nations: {len(primary)}")

Ready. Nations: 2


In [ ]:
# Print data acknowledgement at the top of every notebook
print_data_acknowledgment(source_keys=["usda_ssurgo"])

TRIBAL SOILS AND GEOLOGY — DATA GOVERNANCE ACKNOWLEDGMENT

This analysis uses data that describes the lands and subsurface
resources of the Oceti Sakowin — the Lakota, Dakota, and Nakota
peoples. This data is governed by the following frameworks:

OCAP®  : Tribal Nations have the right to Ownership, Control,
         Access, and Possession of data about their lands,
         including subsurface geological and soil data.
         Reference: https://fnigc.ca/ocap-training/

CARE   : Data use must deliver Collective Benefit to Indigenous
         peoples, respect their Authority to Control, uphold
         Responsibility to communities, and center Ethics.
         Reference: https://www.gida-global.org/care

FAIR   : Data is Findable, Accessible, Interoperable, Reusable.
         FAIR governs technical standards; CARE and OCAP® govern
         the ethical obligations FAIR alone does not address.
         Reference: https://www.go-fair.org/fair-principles/

IEEE 2890-2025 : Recommended Pr

## Load SSURGO Map Units

In [3]:
from src.loaders import load_ssurgo_mapunits, load_ssurgo_components

print("Loading SSURGO map unit polygons...")
mapunits = load_ssurgo_mapunits()

if mapunits.empty:
    print()
    print("SSURGO GDB not found. Download instructions:")
    print("  1. Go to https://websoilsurvey.nrcs.usda.gov/")
    print("  2. Use the Soil Data Downloader")
    print("  3. Download by AREASYMBOL:")
    print("     Pine Ridge: SD113, SD007, SD063")
    print("     Rosebud:    SD121, SD095, SD123, SD055")
    print("  4. Extract GDB files to data/raw/ssurgo/")
else:
    print(f"Map units loaded: {len(mapunits):,} polygons")
    print(f"Columns: {mapunits.columns.tolist()}")
    print(f"CRS: {mapunits.crs}")

Loading SSURGO map unit polygons...

SSURGO GDB not found. Download instructions:
  1. Go to https://websoilsurvey.nrcs.usda.gov/
  2. Use the Soil Data Downloader
  3. Download by AREASYMBOL:
     Pine Ridge: SD113, SD007, SD063
     Rosebud:    SD121, SD095, SD123, SD055
  4. Extract GDB files to data/raw/ssurgo/


C:\Users\gekek\AppData\Local\Temp\ipykernel_45572\3673264419.py:4: UserWarning: No SSURGO GDB found in data/raw/ssurgo/. Download from https://websoilsurvey.nrcs.usda.gov/ using the ESRI Soil Data Downloader. See docs/data_intake_guide.md.
  mapunits = load_ssurgo_mapunits()


In [4]:
# Load components and merge key attributes onto map units
print("Loading SSURGO component data...")
components = load_ssurgo_components()

if not components.empty:
    print(f"Components loaded: {len(components):,} rows")
    print(f"Columns: {components.columns.tolist()}")

    # Key component fields for land management
    key_fields = ["mukey", "drainagecl", "hydgrpdcd", "taxorder",
                  "taxsuborder", "taxgrtgroup", "farmlndcl"]
    available  = [f for f in key_fields if f in components.columns]
    print(f"\nKey fields available: {available}")

    if "drainagecl" in components.columns:
        print(f"\nDrainage class distribution:")
        print(components["drainagecl"].value_counts().to_string())

    if "taxorder" in components.columns:
        print(f"\nSoil order distribution:")
        print(components["taxorder"].value_counts().to_string())

Loading SSURGO component data...


C:\Users\gekek\AppData\Local\Temp\ipykernel_45572\991430673.py:3: UserWarning: No SSURGO GDB found. See load_ssurgo_mapunits docstring.
  components = load_ssurgo_components()


## Soil Attributes and Management Significance

In [ ]:
if not mapunits.empty and not components.empty:
    # Merge dominant component attributes onto map unit polygons
    if "mukey" in components.columns and "mukey" in mapunits.columns:
        # Get dominant component (highest comppct_r) per map unit
        if "comppct_r" in components.columns:
            dom_comp = (components.sort_values("comppct_r", ascending=False)
                        .groupby("mukey").first().reset_index())
        else:
            dom_comp = components.groupby("mukey").first().reset_index()

        mapunits_merged = mapunits.merge(
            dom_comp[["mukey"] + [f for f in
                      ["drainagecl","hydgrpdcd","taxorder","farmlndcl",
                       "drclassdcd","hydgrpdcd"] if f in dom_comp.columns]],
            on="mukey", how="left"
        )
        print(f"Merged: {len(mapunits_merged)} map units with component attributes")

        # Erodibility summary
        if "k factor" in components.columns:
            k_high = components["k factor"].astype(float) >= CONFIG["ssurgo"]["k_factor_high"]
            print(f"\nHigh erodibility (K ≥ {CONFIG['ssurgo']['k_factor_high']}): "
                  f"{k_high.sum()} components ({k_high.mean()*100:.1f}%)")
    else:
        mapunits_merged = mapunits
        print("Note: mukey not found in both tables: cannot merge attributes")
else:
    print("Load SSURGO data to proceed.")

Load SSURGO data to proceed.


In [ ]:
if not mapunits.empty:
    fig, axes = plt.subplots(1, 2, figsize=(15, 8))
    attr_col = "drainagecl" if "drainagecl" in mapunits.columns else None
    if "mapunits_merged" in dir() and attr_col in mapunits_merged.columns:
        plot_gdf = mapunits_merged
    else:
        plot_gdf = mapunits
        attr_col = None

    for ax, (nation, bbox_k) in zip(axes,
        [("Pine Ridge", PINE_RIDGE_BBOX), ("Rosebud", ROSEBUD_BBOX)]):
        from shapely.geometry import box as sbox
        clip_b  = sbox(*bbox_k)
        sub     = plot_gdf[plot_gdf.geometry.intersects(clip_b)].copy()
        nb_gdf  = primary[primary["common_name"].str.contains(nation.split()[0])]

        if not sub.empty:
            if attr_col and attr_col in sub.columns:
                sub.to_crs(CRS_WEB).plot(
                    ax=ax, column=attr_col, categorical=True,
                    alpha=0.8, legend=True, cmap="RdYlGn",
                    legend_kwds={"fontsize":7, "loc":"lower right"}
                )
            else:
                sub.to_crs(CRS_WEB).plot(
                    ax=ax, facecolor=TEAL_LT, edgecolor="white",
                    linewidth=0.3, alpha=0.8
                )

        if not nb_gdf.empty:
            nb_gdf.to_crs(CRS_WEB).plot(
                ax=ax, facecolor="none", edgecolor=TEAL, linewidth=2.5, zorder=5
            )
        try:
            ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, alpha=0.3, zoom=9)
        except Exception: pass
        ax.set_axis_off()
        title_attr = f"\n{attr_col}" if attr_col else ""
        ax.set_title(f"SSURGO Map Units {nation}{title_attr}",
                     fontsize=10, fontweight="bold")

    plt.suptitle("SSURGO Soil Map for Pine Ridge and Rosebud Reservations",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    fig.savefig(FIGURES_DIR/"05_ssurgo_map.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Download SSURGO data and re-run. See cell 1 for instructions.")

Download SSURGO data and re-run. See cell 1 for instructions.


## Farmland Classification and Land Capability

SSURGO includes a farmland classification (`farmlndcl`) that identifies
prime farmland, unique farmland, and farmland of statewide or local
importance. On Pine Ridge and Rosebud, this classification intersects
directly with questions of agricultural land sovereignty, land capable
of food production is central to food sovereignty discussions.

Note that SSURGO farmland classification reflects USDA criteria and does
not incorporate Tribal land management frameworks, traditional land use
patterns, or Indigenous concepts of land capability. The data should be
used alongside Tribal knowledge and governance.

In [ ]:
# Check for farmland classification
if not components.empty and "farmlndcl" in components.columns:
    farm_summary = components["farmlndcl"].value_counts()
    total = farm_summary.sum()
    print("FARMLAND CLASSIFICATION for SSURGO Components")
    for cls, n in farm_summary.items():
        bar = chr(9608) * int(n/total*40)
        print(f"  {str(cls)[:40]:<40}: {n:>5} ({n/total*100:>4.1f}%) {bar}")
    print()
    print("Note: SSURGO farmland classification reflects USDA criteria.")
    print("Review with Tribal natural resource staff for local interpretation.")
else:
    print("farmlndcl not available in component table.")

farmlndcl not available in component table.


In [ ]:
# Print citations
print(generate_citations(["usda_ssurgo","census_aiannh"]))

DATA CITATIONS

USDA NRCS SSURGO — Soil Survey Geographic Database
  Soil Survey Staff, Natural Resources Conservation Service, United States Department of Agriculture. Web Soil Survey. Available online at https://websoilsurvey.nrcs.usda.gov/.
  https://websoilsurvey.nrcs.usda.gov/
  Steward: USDA Natural Resources Conservation Service (NRCS) | License: Public domain

US Census Bureau — TIGER/Line AIANNH Boundaries
  US Census Bureau. TIGER/Line Shapefiles: American Indian / Alaska Native / Native Hawaiian Areas (AIANNH). https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html
  https://www.census.gov/geographies/mapping-files/
  Steward: US Census Bureau | License: Public domain

TERRITORIAL PROVENANCE
  1868 Fort Laramie Treaty — Oceti Sakowin territory, including the Great Sioux Reservation
  The lands of Pine Ridge and Rosebud Reservations are the sovereign territory of the Oglala Lakota and Sicangu Lakota peoples respectively. Federal and state geolog